# B3 — обучение на Google Colab (T4 GPU)

Переобучение **Stage-2 EfficientNet — 48 классов (43 GTSRB + 5 RU)** и опционально **Stage-1 YOLO**.

## Один раз ДО запуска
Залей на Google Drive (`MyDrive/`) архив `C:\mitgo\mitgo_data.zip` **под именем `mitgo_data_v2.zip`**.

> ⚠️ Имя НОВОЕ намеренно. Google Drive не перезаписывает файлы — заливая старое имя
> поверх, получаешь `mitgo_data (1).zip`, а Colab читает старый. Новое имя = нет конфликта.
> Если уже есть `mitgo_data_v2.zip` на Drive — сначала удали его (и Корзину).

Внутри архива (проверяется автоматически ниже): GTSRB Kaggle (`Train.csv`/`Test.csv`/`Train/`/`Test/`),
`rtsd_crops/{train,val}/{43..47}/`, `gtsdb_yolo/` (для YOLO).

Runtime → Change runtime type → **GPU (T4)**, затем ячейки сверху вниз.

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())

In [ ]:
# Clone repo (Stage-2 / tooling). git pull обновляет .py-фиксы (evaluate.py и т.д.).
%cd /content
![ -d MITGOTSD ] || git clone https://github.com/damir095/MITGOTSD.git
%cd /content/MITGOTSD
!git pull --ff-only || true
!git log --oneline -1

In [ ]:
# Deps. У Colab уже стоит CUDA-torch — НЕ переустанавливать torch.
!pip -q install ultralytics timm scikit-learn opencv-python-headless pillow pandas tqdm

In [ ]:
# Mount Drive + ПРЕДПРОВЕРКА архива ДО распаковки (читаем ровно то, что увидит Colab).
from google.colab import drive
drive.mount('/content/drive')

DATA_ZIP  = '/content/drive/MyDrive/mitgo_data_v2.zip'   # <-- имя на Drive
DATA_ROOT = '/content/datasets'

import zipfile, os
assert os.path.exists(DATA_ZIP), f'НЕТ файла на Drive: {DATA_ZIP} — проверь имя/папку'
_z  = zipfile.ZipFile(DATA_ZIP)
_nm = [x.replace(chr(92), '/') for x in _z.namelist()]
_has = lambda p: any(p in x for x in _nm)
print('размер архива :', f'{os.path.getsize(DATA_ZIP)/1e6:.0f} MB (ожид. ~587)')
print('entries       :', len(_nm), '(ожид. 84334)')
print('Train.csv     :', any(x.endswith('Train.csv') for x in _nm))
print('gtsdb.yaml    :', any(x.endswith('gtsdb.yaml') for x in _nm))
for c in (43, 44, 45, 46, 47):
    print(f'rtsd cls {c}    :', _has(f'rtsd_crops/train/{c}/'))
assert all(_has(f'rtsd_crops/train/{c}/') for c in range(43, 48)), (
    'СТАРЫЙ/НЕ ТОТ архив на Drive (нет классов 46/47). Перезалей mitgo_data_v2.zip.')
print('\nOK — архив правильный, можно распаковывать.')

In [ ]:
# Чистая распаковка (всегда с нуля) + проверка классов + авто-пути.
import shutil, pathlib, glob
shutil.rmtree(DATA_ROOT, ignore_errors=True)
pathlib.Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
_z.extractall(DATA_ROOT)
print('распаковано в', DATA_ROOT)

cls = sorted(int(pathlib.Path(p).name) for p in
             glob.glob(f'{DATA_ROOT}/**/rtsd_crops/train/*', recursive=True)
             if pathlib.Path(p).is_dir())
print('rtsd_crops train classes:', cls)
assert cls == [43, 44, 45, 46, 47], f'ожидались 43-47, получено {cls}'

train_csv = glob.glob(f'{DATA_ROOT}/**/Train.csv', recursive=True)
rtsd_dir  = glob.glob(f'{DATA_ROOT}/**/rtsd_crops', recursive=True)
yaml_path = glob.glob(f'{DATA_ROOT}/**/gtsdb.yaml', recursive=True)
assert train_csv and rtsd_dir, 'Train.csv / rtsd_crops не найдены после распаковки'
os.environ['GTSRB_KAGGLE_ROOT'] = str(pathlib.Path(train_csv[0]).parent)
os.environ['RTSD_CROPS_ROOT']   = rtsd_dir[0]
print('GTSRB_KAGGLE_ROOT =', os.environ['GTSRB_KAGGLE_ROOT'])
print('RTSD_CROPS_ROOT   =', os.environ['RTSD_CROPS_ROOT'])
print('gtsdb.yaml        =', yaml_path[0] if yaml_path else '(нет — пропусти YOLO)')

## Stage-2 — EfficientNet (48 классов)
Сначала smoke (1+1 эпоха) — проверить, что цепочка цела, потом полный прогон.

In [ ]:
# Smoke: должен дойти до отчёта GTSRB-теста без падения (фикс evaluate.py).
!cd /content/MITGOTSD && GTSRB_EPOCHS_HEAD=1 GTSRB_EPOCHS_FULL=1 GTSRB_BATCH_SIZE=64 python train.py

In [ ]:
# Полный прогон. T4 16 GB -> большой батч. ~2 часа.
!cd /content/MITGOTSD && GTSRB_BATCH_SIZE=256 GTSRB_NUM_WORKERS=2 python train.py

In [ ]:
# ПРОВЕРКА RU-классов 43-47 на rtsd_crops/val ДО сохранения.
# evaluate.py/train.py меряют только немецкие классы — дыру в 46/47 не покажут.
import glob, torch
from PIL import Image
from src.camera import _transform, CLASS_NAMES
from src.model import build_model

m = build_model(freeze_backbone=False)
m.load_state_dict(torch.load('experiments/checkpoints/best.pt', map_location='cpu'))
m.eval()
bad = []
for c in (43, 44, 45, 46, 47):
    fs = glob.glob(f'/content/datasets/**/rtsd_crops/val/{c}/*.jpg', recursive=True)[:80]
    assert fs, f'НЕТ val-кропов класса {c} — архив без этого класса'
    ok = 0
    with torch.no_grad():
        for f in fs:
            x = _transform(Image.open(f).convert('RGB')).unsqueeze(0)
            ok += int(m(x).argmax(1).item() == c)
    acc = ok / len(fs)
    flag = 'OK' if acc >= 0.5 else 'ПЛОХО'
    print(f'[{c}] {CLASS_NAMES[c]:<20} {ok}/{len(fs)} = {acc:.3f}  {flag}')
    if acc < 0.5:
        bad.append(c)
assert not bad, f'Классы {bad} не обучились — НЕ сохраняй, разбирайся с данными'
print('\nВсе 5 RU-классов OK — можно сохранять.')

In [ ]:
# Сохранить 48-классовый классификатор на Drive + скачать.
import shutil
src = '/content/MITGOTSD/experiments/checkpoints/best.pt'
shutil.copy(src, '/content/drive/MyDrive/efficientnet_ru48.pt')
print('на Drive: efficientnet_ru48.pt')
from google.colab import files; files.download(src)

## Stage-1 — YOLO (опционально)
Рабочий детектор Colab-640 уже есть локально — переобучай **только если нужно**.
`gtsdb.yaml` содержит Windows-путь — чиним под Colab.

In [ ]:
import pathlib
assert yaml_path, 'gtsdb.yaml не найден — нужен gtsdb_yolo/ в архиве'
yp = yaml_path[0]
lines = pathlib.Path(yp).read_text().splitlines()
lines = [f'path: {pathlib.Path(yp).parent.as_posix()}' if l.startswith('path:') else l
         for l in lines]
pathlib.Path(yp).write_text(chr(10).join(lines) + chr(10))
print(pathlib.Path(yp).read_text())

In [ ]:
!cd /content/MITGOTSD && yolo detect train model=yolov8n.pt data={yp} \
    imgsz=640 batch=32 epochs=100 device=0 project=experiments/yolo name=gtsdb

In [ ]:
import shutil, glob
bp = glob.glob('/content/MITGOTSD/runs/detect/**/best.pt', recursive=True)[0]
shutil.copy(bp, '/content/drive/MyDrive/yolo_gtsdb.pt')
print('на Drive: yolo_gtsdb.pt  (из', bp, ')')
from google.colab import files; files.download(bp)

## Назад на локальную машину
Скачанный **`efficientnet_ru48.pt`** положи в `project/` — дальше локально:
`efficientnet_ru48.pt` → `project/experiments/checkpoints/best.pt`,
(если переобучал YOLO) `yolo_gtsdb.pt` → `project/runs/detect/experiments/yolo/gtsdb/weights/best.pt`,
затем `python -m src.evaluate_b3` и `python -m src.yolo_pipeline`.